# Quick Smoke Test: Hypothesis-Testing Pipeline

**Purpose:** Runs a rapid 5-second end-to-end verification of the hypothesis-testing pipeline using the small test dataset `TEST`.

Use this notebook on Kaggle or locally before launching large runs to verify that all imports, configurations, null rewiring, and exports work properly.


In [ ]:
# Cell 1: Setup sys.path and environment
# ============================================================
import os, sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    for p in [Path('/kaggle/input/datasets/jeet7771/flywire-codebase'), Path('/kaggle/working'), Path('/kaggle/input/flywire-codebase')]:
        if p.exists() and str(p) not in sys.path:
            sys.path.insert(0, str(p))
            print(f'[OK] Codebase path: {p}')
            break
else:
    repo_root = Path(os.getcwd()).resolve()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    print(f'[OK] Local repo root: {repo_root}')

print('[OK] Cell 1 complete.')


In [ ]:
# Cell 2: Framework Imports
# ============================================================
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

from hypothesis_testing.config import HypothesisExperimentConfig, ExecutionMode
from hypothesis_testing.runners.hypothesis_experiment_runner import HypothesisExperimentRunner
from hypothesis_testing.runners.compare_existing_runner import CompareExistingRunner

print('[OK] All hypothesis-testing modules imported successfully.')


In [ ]:
# Cell 3: Fast Test Configuration (~2 seconds runtime)
# ============================================================
DATASET_ROOT = '/kaggle/input/datasets/jeet7771/flywire-all-datasets' if IS_KAGGLE else 'research_data/raw'
CONFIGS_ROOT = 'configs'
OUTPUT_ROOT = Path('results') / 'smoke_test'

test_config = HypothesisExperimentConfig(
    dataset_name='TEST',
    dataset_root=DATASET_ROOT,
    configs_root=CONFIGS_ROOT,
    execution_mode='null_only',
    null_model_name='degree_preserving',
    null_graph_seeds=[1],
    random_seeds=[1, 2],
    error_model_names=['missed_synapses', 'split_errors'],
    error_rates=[0.0, 0.05],
    analysis_names=['basic_structure'],
    output_root=str(OUTPUT_ROOT),
)

print(f'[OK] Smoke test config ready: Mode={test_config.execution_mode.value}, Dataset={test_config.dataset_name}')


In [ ]:
# Cell 4: Run Smoke Test
# ============================================================
runner = HypothesisExperimentRunner()
result = runner.run(test_config)

print(f'Pipeline Status : {result.status}')
print(f'Execution Time  : {result.runtime_seconds:.2f}s')

null_csv = OUTPUT_ROOT / 'TEST' / 'null_observations' / 'replicate_level_effects.csv'
if null_csv.exists():
    df = pd.read_csv(null_csv)
    print(f'[SUCCESS] {len(df)} replicate records produced!')
    display(df.head(6))
else:
    print('[ERROR] Replicate CSV not found.')
